In [1]:
import pandas as pd
import numpy as np
#from sklearn.ensemble import IsolationForest

In [2]:
# ==========================
# CONFIGURAÇÕES
# ==========================
input_file = "dataset_com_ciclos.xlsx"
output_file = "cycles_features.xlsx"

stroke_total_m = 0.10  # curso total do pistão em metros
velocidade_m_s = 1.0   # velocidade do pistão em m/s

# Calcula o tempo de ida e volta
tempo_ida_s = stroke_total_m / velocidade_m_s
tempo_volta_s = tempo_ida_s
tempo_ciclo_s = tempo_ida_s + tempo_volta_s

print(f"Tempo de ida: {tempo_ida_s:.3f} s")
print(f"Tempo de volta: {tempo_volta_s:.3f} s")
print(f"Tempo total do ciclo: {tempo_ciclo_s:.3f} s")

Tempo de ida: 0.100 s
Tempo de volta: 0.100 s
Tempo total do ciclo: 0.200 s


In [3]:
# ==========================
# CARREGAR DADOS
# ==========================
df = pd.read_excel(input_file)
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [4]:
# ==========================
# CALCULAR TEMPOS, VELOCIDADES MÉDIAS E MARGEM DE ERRO POR CICLO
# ==========================
v_esperada = 1.0  # Velocidade nominal do atuador em m/s
resultados = []

for ciclo, grupo in df.groupby("Ciclo"):
    if ciclo > 0:  # ignora linhas fora de ciclos
        # Ordena o grupo por timestamp
        grupo = grupo.sort_values("timestamp")
        
        # Tempo total do ciclo registrado
        inicio = grupo["timestamp"].iloc[0]
        fim = grupo["timestamp"].iloc[-1]
        duracao_total = (fim - inicio).total_seconds()

        # --------------------------
        # tempo de avanço (ida)
        # --------------------------
        t_avanco = None
        speed_adv = None
        error_adv = None
        try:
            if "Avancado_1S2" in grupo.columns:
                t_ini_avanco = grupo.loc[grupo["V1_12"] == 1, "timestamp"].iloc[0]
                t_fim_avanco = grupo.loc[grupo["Avancado_1S2"] == 1, "timestamp"].iloc[0]
                t_avanco = (t_fim_avanco - t_ini_avanco).total_seconds()
                if t_avanco > 0:
                    speed_adv = stroke_total_m / t_avanco
                    error_adv = ((speed_adv - v_esperada) / v_esperada) * 100
        except IndexError:
            pass

        # --------------------------
        # tempo de recuo (volta)
        # --------------------------
        t_recuo = None
        speed_rec = None
        error_rec = None
        try:
            if "Recuado_1S1" in grupo.columns:
                t_ini_recuo = grupo.loc[grupo["V1_14"] == 1, "timestamp"].iloc[0]
                t_fim_recuo = grupo.loc[grupo["Recuado_1S1"] == 1, "timestamp"].iloc[0]
                t_recuo = ((t_fim_recuo - t_ini_recuo)*(-1)).total_seconds()
                if t_recuo > 0:
                    speed_rec = stroke_total_m / t_recuo
                    error_rec = ((speed_rec - v_esperada) / v_esperada) * 100
        except IndexError:
            pass

        # --------------------------
        # adicionar resultados
        # --------------------------
        resultados.append({
            "Ciclo": ciclo,
            "cycle_time_s": duracao_total,
            "t_avanco_s": t_avanco,
            "speed_adv_m_s": speed_adv,
            #"error_adv_percent": error_adv,
            "t_recuo_s": t_recuo,
            "speed_rec_m_s": speed_rec,
            #"error_rec_percent": error_rec
        })

# Criar DataFrame final
df_cycles = pd.DataFrame(resultados)

# Exibir resultados
df_cycles.head()


,Ciclo,cycle_time_s,t_avanco_s,speed_adv_m_s,t_recuo_s,speed_rec_m_s
0,1,2.489,1.230,0.081301,1.053,0.094967
1,2,2.721,1.214,0.082372,1.029,0.097182
2,3,2.403,1.213,0.082440,0.815,0.122699
3,4,2.567,1.275,0.078431,1.119,0.089366
4,5,2.777,1.405,0.071174,1.027,0.097371


In [5]:
# ==========================
# SALVAR RESULTADOS
# ==========================
df_cycles.to_excel(output_file, index=False)

print(f"✅ Resultados salvos em {output_file}")
print(df_cycles.head(10))

✅ Resultados salvos em cycles_features.xlsx
   Ciclo  cycle_time_s  t_avanco_s  speed_adv_m_s  t_recuo_s  speed_rec_m_s
0      1         2.489       1.230       0.081301      1.053       0.094967
1      2         2.721       1.214       0.082372      1.029       0.097182
2      3         2.403       1.213       0.082440      0.815       0.122699
3      4         2.567       1.275       0.078431      1.119       0.089366
4      5         2.777       1.405       0.071174      1.027       0.097371
5      6         2.529       1.048       0.095420      0.889       0.112486
6      7         2.446       1.201       0.083264      0.815       0.122699
7      8         2.634       1.186       0.084317      1.010       0.099010
8      9         2.447       1.200       0.083333      1.027       0.097371
9     10         2.649       1.260       0.079365      1.136       0.088028
